# Hotel Review Intelligence Engine — Demo Notebook

**Expedia Group Campus Hackathon 2026 — Innovation Round**

This notebook loads the already-processed outputs from `pipeline.py`, `drift_analysis.py`, and `recommend.py`, and walks through the three core capabilities of the system:

1. Aspect-based sentiment scoring
2. Temporal / seasonal performance tracking
3. Personalized, evidence-based recommendations

No models are re-run here — this notebook is a fast, lightweight walkthrough of results already generated. Run `python pipeline.py`, `python drift_analysis.py`, and `python recommend.py` first if the files under `data/processed/` don't exist yet.

In [ ]:
import pandas as pd
import json
from IPython.display import display, Image, Markdown

pd.set_option('display.max_colwidth', None)

## 1. Aspect-Based Sentiment — Raw Output

This is the output of `pipeline.py`: signed sentiment scores per hotel, per aspect, per month.

In [ ]:
aspect_sentiment = pd.read_csv('data/processed/aspect_sentiment.csv')
print(f"Shape: {aspect_sentiment.shape}")
aspect_sentiment.head(10)

### Quick sanity check: sentiment sign distribution

Confirms that both strongly positive and strongly negative scores are present — i.e. the model isn't defaulting to one label.

In [ ]:
print("Score summary statistics:")
print(aspect_sentiment['avg_sentiment_score'].describe())
print(f"\nPositive-average rows: {(aspect_sentiment['avg_sentiment_score'] > 0).sum()}")
print(f"Negative-average rows: {(aspect_sentiment['avg_sentiment_score'] < 0).sum()}")

## 2. Temporal & Seasonal Performance Tracking

This is the output of `drift_analysis.py`: which hotel-aspect pairs are trending up, down, or stable, and whether any season stands out.

In [ ]:
performance_summary = pd.read_csv('data/processed/hotel_performance_summary.csv')
print(f"Shape: {performance_summary.shape}")

print("\nTrend direction counts:")
print(performance_summary['trend_direction'].value_counts())

print("\nHotel-aspect pairs with a flagged anomalous season:")
print(performance_summary['flagged_season'].notna().sum(), "out of", len(performance_summary))

In [ ]:
# Show the strongest declining trends — these are the hotels/aspects most worth flagging
performance_summary.sort_values('trend_slope').head(10)

### Sample trend visualization

Pre-generated chart from `drift_analysis.py` showing a hotel's aspect sentiment over time.

In [ ]:
import os

plots_dir = 'data/processed/plots'
available_plots = os.listdir(plots_dir) if os.path.exists(plots_dir) else []
print("Available plots:", available_plots)

if available_plots:
    display(Image(filename=os.path.join(plots_dir, available_plots[0])))

## 3. Personalized, Evidence-Based Recommendations

This is the output of `recommend.py`: for each of the 50 user profiles, the top 5 recommended hotels with relevance scores and plain-language evidence.

In [ ]:
with open('data/processed/recommendations.json') as f:
    recommendations = json.load(f)

print(f"Total profiles: {len(recommendations)}")
print(f"Sample profile IDs: {list(recommendations.keys())[:5]}")

### Pick a profile to inspect

Change `profile_id` below to explore a different persona (P01 through P50).

In [ ]:
profile_id = "P01"

profile_data = recommendations[profile_id]

print(f"Profile: {profile_id}")
print(f"Archetype: {profile_data['archetype']}")
print(f"Desired dimensions: {profile_data['desired_dims']}")
print("\nTop 5 recommended hotels:\n")

top_hotels_df = pd.DataFrame(profile_data['top_hotels'])
top_hotels_df

### Readable version — recommendation card style

In [ ]:
md_output = f"### Recommendations for `{profile_id}` — *{profile_data['archetype']}*\n\n"
md_output += f"**Priorities:** {', '.join(profile_data['desired_dims'])}\n\n"

for hotel in profile_data['top_hotels']:
    md_output += (
        f"**#{hotel['rank']} — {hotel['hotel_name']}** "
        f"(relevance: {hotel['relevance_score']})\n\n"
        f"> {hotel['evidence']}\n\n"
    )

display(Markdown(md_output))

### Compare two different personas

This demonstrates that recommendations genuinely differ based on inferred priorities — e.g. a corporate traveler vs. a family traveler should surface different top hotels.

In [ ]:
compare_ids = ["P02", "P03"]  # corporate road-warrior vs. family with toddlers

for pid in compare_ids:
    data = recommendations[pid]
    print(f"\n=== {pid} — {data['archetype']} ===")
    print(f"Priorities: {data['desired_dims']}")
    print("Top 3 hotels:")
    for hotel in data['top_hotels'][:3]:
        print(f"  #{hotel['rank']} {hotel['hotel_name']} (score: {hotel['relevance_score']}) — {hotel['evidence']}")

## Summary

This notebook demonstrates the end-to-end output of the Hotel Review Intelligence Engine:

- **Aspect-level sentiment** extracted from 50,000 reviews across 5 aspects
- **Temporal/seasonal performance tracking** per hotel and aspect
- **Personalized, evidence-based top-5 recommendations** for 50 distinct traveler profiles

See `README.md` for full architecture details, key dataset findings, assumptions, and limitations.